# 01 — Data Preparation
**Goal:** Load raw data, fix types, add useful features, save clean file.

Input : `data/petrol_pump_data_2024_2026.xlsx`  
Output: `data/clean_data.csv`

In [1]:
# Install once if needed
# !pip install pandas openpyxl matplotlib seaborn scikit-learn imbalanced-learn xgboost

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ')

Libraries loaded 


In [8]:
# ── Load data ──────────────────────────────────────────────────
FILE_PATH = '../data/petrol_pump_data_2024_2026.xlsx'

df = pd.read_excel(FILE_PATH)

print(f'Rows    : {df.shape[0]}')
print(f'Columns : {df.shape[1]}')
print(f'\nColumns: {df.columns.tolist()}')
df.head()
df.describe()

Rows    : 806
Columns : 14

Columns: ['Date', 'Day', 'Opening_Stock', 'MS_Sold', 'HSD1_Sold', 'HSD2_Sold', 'HSD3_Sold', 'Total_Sold', 'Closing_Stock', 'Cash', 'Online', 'Card', 'Dip', 'Refill_Required']


,Opening_Stock,MS_Sold,HSD1_Sold,HSD2_Sold,HSD3_Sold,Total_Sold,Closing_Stock,Cash,Online,Card,Dip
count,806.000000,802.000000,805.000000,802.000000,802.000000,803.000000,805.00000,806.000000,806.000000,806.000000,806.000000
mean,8008.878412,567.694514,1804.443478,1428.568579,952.825436,4756.148194,3639.64472,196524.568238,149630.155087,101815.571960,30.256824
std,3527.340993,102.365006,273.036520,224.270377,184.976597,691.529651,3044.65015,31801.621363,25296.320300,18633.611199,24.944605
min,2020.000000,315.000000,1155.000000,789.000000,451.000000,2835.000000,0.00000,114051.000000,83302.000000,53570.000000,1.000000
25%,5516.000000,497.000000,1602.000000,1268.500000,818.000000,4278.000000,88.00000,175022.000000,131713.000000,88476.500000,1.000000
50%,7538.000000,556.500000,1796.000000,1414.500000,934.000000,4698.000000,3055.00000,194132.500000,147456.000000,100195.000000,25.000000
75%,12000.000000,634.000000,1962.000000,1579.000000,1085.750000,5179.500000,6911.00000,217883.250000,164925.500000,113571.000000,57.000000
max,12000.000000,913.000000,2714.000000,2147.000000,1564.000000,6869.000000,9165.00000,303203.000000,234398.000000,163590.000000,76.000000


In [9]:
# ── Fix data types & handle missing values ─────────────────────
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

missing = df.isnull().sum().sum()
print(f'Missing values: {missing}')
if missing > 0:
    df.fillna(method='ffill', inplace=True)
    print('Filled with forward-fill')
else:
    print('No missing values')

Missing values: 17
Filled with forward-fill


In [10]:
# ── Add date features ──────────────────────────────────────────
df['Year']      = df['Date'].dt.year
df['Month']     = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek   # 0=Monday, 6=Sunday
df['Quarter']   = df['Date'].dt.quarter
df['Is_Weekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)

# Day name → number
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
df['Day_Num'] = df['Day'].map({d: i for i, d in enumerate(day_order)})

# Target: 1 = Refill needed, 0 = No refill
df['Target'] = (df['Refill_Required'] == 'Yes').astype(int)

print('Date features added ')
df[['Date','Year','Month','DayOfWeek','Is_Weekend','Target']].head()

Date features added 


,Date,Year,Month,DayOfWeek,Is_Weekend,Target
0,2024-01-01,2024,1,0,0,0
1,2024-01-02,2024,1,1,0,0
2,2024-01-03,2024,1,2,0,1
3,2024-01-04,2024,1,3,0,0
4,2024-01-05,2024,1,4,0,0


In [6]:
# ── Add business features ──────────────────────────────────────
# How much of the tank is left (0 = empty, 1 = full)
df['Stock_Ratio'] = (df['Closing_Stock'] / 12000).round(4)

# Average sales over last 7 days — captures weekly trend
df['Rolling_7d_Sales'] = df['Total_Sold'].rolling(window=7, min_periods=1).mean().round(0)

# Previous day's closing stock (how full were we yesterday?)
df['Prev_Closing'] = df['Closing_Stock'].shift(1).fillna(12000)

# Seasonal flags
df['Is_Festival_Month'] = df['Month'].isin([10, 11]).astype(int)  # Oct-Nov
df['Is_Monsoon_Month']  = df['Month'].isin([6, 7, 8]).astype(int) # Jun-Aug

print(f'Total features: {df.shape[1]}')
print(df.columns.tolist())

Total features: 26
['Date', 'Day', 'Opening_Stock', 'MS_Sold', 'HSD1_Sold', 'HSD2_Sold', 'HSD3_Sold', 'Total_Sold', 'Closing_Stock', 'Cash', 'Online', 'Card', 'Dip', 'Refill_Required', 'Year', 'Month', 'DayOfWeek', 'Quarter', 'Is_Weekend', 'Day_Num', 'Target', 'Stock_Ratio', 'Rolling_7d_Sales', 'Prev_Closing', 'Is_Festival_Month', 'Is_Monsoon_Month']


In [7]:
# ── Summary & Save ─────────────────────────────────────────────
print('=== Final Dataset Summary ===')
print(f'Rows       : {len(df)}')
print(f'Columns    : {df.shape[1]}')
print(f'Date range : {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'Refill days: {df["Target"].sum()} ({df["Target"].mean()*100:.1f}%)')
print(f'No refill  : {(df["Target"]==0).sum()} ({(df["Target"]==0).mean()*100:.1f}%)')

df.to_csv('../data/clean_data.csv', index=False)
print('\n Saved: data/clean_data.csv')

=== Final Dataset Summary ===
Rows       : 806
Columns    : 26
Date range : 2024-01-01 → 2026-03-16
Refill days: 303 (37.6%)
No refill  : 503 (62.4%)

 Saved: data/clean_data.csv
